# 03 – Explorative Datenanalyse (EDA)

**CRISP-DM-Phase:** Data Understanding (vertieft) / Vorbereitung der Modellierung.

**Methodischer Vorbehalt (wichtig!):** EDA ist **hypothesengenerierend**, nicht
-bestätigend. Alle hier sichtbaren Zusammenhänge sind **korrelativ**, nicht kausal,
und stammen (im Demo-Modus) aus **synthetischen** Daten. Für belastbare Aussagen sind
die echten Daten und die spätere Modellevaluation maßgeblich.

Zu **jeder** Grafik beantworten wir: *Was zeigt sie? Warum ist sie relevant? Welche
Interpretation ist möglich? Welche Grenzen hat sie?*

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, utils, eda

feat = utils.load_processed("feature_matrix")
print("Feature-Matrix:", feat.shape)

Feature-Matrix: (307511, 161)


## 1. Verteilung der Zielvariable

**Was:** Häufigkeit von Ausfall (1) vs. kein Ausfall (0).
**Warum:** Bestimmt Metrik- und Trainingsstrategie.
**Interpretation:** Bei ~10 % Positiven ist Accuracy irreführend (ein Trivialmodell
erreicht ~90 %). **Grenzen:** Randverteilung ohne Merkmalsbezug.

In [2]:
fig = eda.plot_target_distribution(feat); plt.show()

17:56:26 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\target_distribution.png
C:\Users\annis\AppData\Local\Temp\ipykernel_35880\4060539775.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_target_distribution(feat); plt.show()


## 2. Fehlende Werte

**Was:** Spalten mit höchstem Fehlwertanteil.
**Warum:** Steuert Imputation/Feature-Auswahl.
**Interpretation:** Externe Scores (`EXT_SOURCE_*`) fehlen häufig – Fehlen kann selbst
informativ sein (`missing_values_count` als Feature). **Grenzen:** Mechanismus
(MCAR/MAR/MNAR) bleibt unbekannt.

In [3]:
fig = eda.plot_missingness(feat, top=20); plt.show()

17:56:26 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\missingness.png
C:\Users\annis\AppData\Local\Temp\ipykernel_35880\3915583392.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_missingness(feat, top=20); plt.show()


## 3. Verteilungen zentraler numerischer Features

**Was:** Histogramme von Einkommen, Kredit, Alter, Ratios, externem Score, Verzugsquote.
**Warum:** Schiefe/Ausreißer erkennen.
**Interpretation:** Geldgrößen sind typischerweise stark rechtsschief -> robuste
Skalierung/Transformation sinnvoll. **Grenzen:** univariat, kein Target-Bezug.

In [4]:
num_features = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "age_years",
                "credit_income_ratio", "external_score_mean", "INST_LATE_PAYMENT_RATIO"]
fig = eda.plot_numeric_distributions(feat, num_features); plt.show()

17:56:28 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\numeric_distributions.png
C:\Users\annis\AppData\Local\Temp\ipykernel_35880\1430031709.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_numeric_distributions(feat, num_features); plt.show()


## 4. Ausfall vs. Nicht-Ausfall (Gruppenvergleich)

**Was:** Boxplots zentraler Features getrennt nach TARGET.
**Warum:** Erste Hinweise auf prädiktive Merkmale.
**Interpretation:** Unterscheiden sich Mediane/Streuungen deutlich (z. B. niedrigerer
externer Score in der Ausfallgruppe), ist das ein Relevanzhinweis. **Grenzen:**
bivariat, ohne Wechselwirkungen; rein korrelativ.

In [5]:
cmp_features = ["external_score_mean", "credit_income_ratio", "age_years",
                "annuity_income_ratio", "AMT_CREDIT", "INST_LATE_PAYMENT_RATIO"]
fig = eda.plot_target_comparison(feat, cmp_features); plt.show()

17:56:31 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\target_comparison_boxplots.png
C:\Users\annis\AppData\Local\Temp\ipykernel_35880\1689097759.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_target_comparison(feat, cmp_features); plt.show()


## 5. Korrelationen

**Was:** Heatmap der 15 am stärksten mit TARGET (linear) korrelierten Features +
Rangliste.
**Warum:** Identifiziert lineare Zusammenhänge und Multikollinearität.
**Interpretation:** Hohe |Korrelation| mit TARGET = Kandidat für Prädiktor; hohe
Korrelation *zwischen* Features = Redundanz (relevant für lineare Modelle).
**Grenzen:** Pearson erfasst nur lineare Zusammenhänge; Korrelation ≠ Kausalität.

In [6]:
fig, corr = eda.plot_correlation_heatmap(feat); plt.show()
tbl = eda.target_correlation_table(feat, top=15)
utils.save_table(tbl, "target_correlations", subdir="eda")
tbl

17:56:44 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\correlation_heatmap.png
C:\Users\annis\AppData\Local\Temp\ipykernel_35880\1780713247.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig, corr = eda.plot_correlation_heatmap(feat); plt.show()
17:56:57 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\eda\target_correlations.csv


,feature,abs_corr,corr
0,external_score_mean,0.222052,-0.222052
1,external_score_max,0.196876,-0.196876
2,external_score_min,0.185266,-0.185266
3,EXT_SOURCE_3,0.178919,-0.178919
4,EXT_SOURCE_2,0.160472,-0.160472
5,EXT_SOURCE_1,0.155317,-0.155317
6,BUREAU_MEAN_DAYS_CREDIT,0.089729,0.089729
7,DAYS_BIRTH,0.078239,0.078239
8,age_years,0.078234,-0.078234
9,BUREAU_MIN_DAYS_CREDIT,0.075248,0.075248


## 6. Erste Hypothesen (datengestützt, vorläufig)

Aus der EDA (Demo-Daten) lassen sich u. a. folgende **Hypothesen** ableiten – als
Eingangspunkt für die Modellierung, **nicht** als gesicherte Befunde:

1. **Externe Scores** sind die stärksten (negativen) Korrelate des Ausfalls –
   höhere Bonitätsscores gehen mit geringerem Risiko einher.
2. Eine hohe **Kredit-Einkommens-Relation** (`credit_income_ratio`) erhöht tendenziell
   das Risiko (Schuldentragfähigkeit).
3. **Zahlungsverhalten** (`INST_LATE_PAYMENT_RATIO`) könnte zusätzliche, von den Scores
   unabhängige Information liefern.

**Grenzen aller Hypothesen:** korrelativ, datensatzspezifisch, im Demo-Modus
synthetisch. Bestätigung erfolgt erst über Modell + Interpretation (Notebooks 04/06).

---
### Quellen (im Theorieteil vollständig)
- Tukey, J. W. (1977). *Exploratory Data Analysis.* Addison-Wesley.
- He, H., & Garcia, E. A. (2009). Learning from imbalanced data. *IEEE TKDE.*